In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch

# 1. Identify your specific environment
torch_ver = torch.__version__.split('+')[0]
cuda_ver = torch.version.cuda.replace('.', '')

print(f"Installing for Torch {torch_ver} and CUDA {cuda_ver}...")

# 2. Install the four "hard" dependencies from the official PyG wheel index
!pip install --quiet torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver}.html

# 3. Install/Reinstall the main libraries
!pip install --quiet torch-geometric ogb

In [1]:
import torch
import os.path as osp
from ogb.nodeproppred import PygNodePropPredDataset
import torch_geometric.transforms as T

# A more robust monkeypatch
_orig_load = torch.load

def smart_load(*args, **kwargs):
    # Ensure weights_only is False, even if it's already in kwargs
    kwargs['weights_only'] = False
    return _orig_load(*args, **kwargs)

torch.load = smart_load

try:
    # 1. Define transforms
    # ToUndirected makes the graph symmetric, which helps GCNs learn better
    transform = T.Compose([T.ToSparseTensor(), T.ToUndirected()])
    
    # 2. Load the dataset
    dataset = PygNodePropPredDataset(name='ogbn-arxiv', transform=transform)
    data = dataset[0]
    
    # 3. Get the splits provided by OGB
    split_idx = dataset.get_idx_split()
    train_idx = split_idx['train']
    valid_idx = split_idx['valid']
    test_idx = split_idx['test']
    
    print("✅ Success! Dataset loaded and splits ready.")
    print(f"Graph stats: {data.num_nodes} nodes, {data.num_edges} edges")
    print(f"Training nodes: {train_idx.numel()}")
finally:
    # Restore the original function to keep the rest of your notebook 'clean'
    torch.load = _orig_load

✅ Success! Dataset loaded and splits ready.
Graph stats: 169343 nodes, 1166243 edges
Training nodes: 90941
